# Webscraper for Soil Records

Having cleaned and merged the datasets we had, the next course of action, which we'll execute in this notebook will be to ensure that the missing soil data is populated before we can embark on performing required analysis to answer project objectives.

## Imports and Setup

As is best practice, we are going to first import necessary libraries to help us execute our codes

In [1]:
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

In [2]:
from pathlib import Path

# Defining path to folder
data_path = Path(r"C:\Users\admin\Documents\Projects\maize-yield-optimization-rwanda\data\raw")

def load_csv (filename):
    return pd.read_csv(data_path/filename)

In [ ]:
soil_df = load_csv("Soil_PL.csv")

soil_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 736 entries, 0 to 735
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   plot_id  736 non-null    str    
 1   pH       734 non-null    float64
 2   tot_N    0 non-null      float64
 3   org_C    0 non-null      float64
 4   sand     0 non-null      float64
dtypes: float64(4), str(1)
memory usage: 28.9 KB


: 

We'll begin by populting the total nitrogen soil content (tot_N) field of the Soil Dataframe

We'll start by initiating the data importation process from available web API.

In [ ]:
# =========================
# CONFIGURATION
# =========================
API_URL = "https://rest.isric.org/soilgrids/v2.0/properties/query"

MAX_WORKERS = 10   # Number of parallel threads (controls speed vs API load)
TIMEOUT = 10       # Max seconds to wait for API response
RETRIES = 3        # Number of retry attempts per request

At this point what we want to do is query the SoilGrids API for soil organic carbon (SOC) data.

However, at upto this point, our dataset lacks latitude and longitude records, so it may just be impossible to input the records we want.

So, we are goint to temporarily append the latitude and longitude records from Field dataset. This way we'll be able to add Soil Organic composition data  at respective latitudes and longitude

In [ ]:
# =========================
# FUNCTION: FETCH SOIL ORGANIC CARBON (SOC)
# =========================
def fetch_soc(lat, lon, depth="0-5cm"):
    """
    Queries SoilGrids API for soil organic carbon (SOC)
    at a given latitude and longitude.

    Returns:
        float (mean SOC value) or None if unavailable
    """

    # Define API query parameters
    params = {
        "lat": lat,
        "lon": lon,
        "property": "soc",   # Soil Organic Carbon
        "depth": depth
    }

    # Retry logic to improve robustness
    for attempt in range(RETRIES):
        try:
            # Send GET request with timeout protection
            response = requests.get(API_URL, params=params, timeout=TIMEOUT)

            # If response is not successful, retry
            if response.status_code != 200:
                continue

            # Convert response to JSON
            data = response.json()

            # =========================
            # EXTRACTION LOGIC
            # =========================
            # Navigate nested structure:
            # properties → layers → depths → values → mean
            layers = data.get("properties", {}).get("layers", [])

            for layer in layers:
                # Ensure we are extracting the correct property (soc)
                if layer.get("name") == "soc":
                    depths = layer.get("depths", [])

                    if depths:
                        values = depths[0].get("values", {})
                        return values.get("mean")  # Extract numeric value

            # If structure exists but no value found
            return None

        except requests.exceptions.RequestException:
            # Handle network-related errors and retry
            time.sleep(1)

    # If all retries fail, return None
    return None



In [ ]:

# =========================
# FUNCTION: FILL MISSING org_C VALUES
# =========================
def fill_orgC_column(df):
    """
    Fills missing values in the 'org_C' column using
    SoilGrids API data.

    Only missing values are queried to improve efficiency.
    """

    # Start with existing values (to preserve non-null entries)
    results = df["org_C"].tolist()

    # ThreadPoolExecutor enables parallel API requests
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

        # Map each future (task) to its dataframe index
        future_to_index = {}

        for idx, row in df.iterrows():

            # Only query API if value is missing
            if pd.isna(row["org_C"]):
                future = executor.submit(fetch_soc, row["lat"], row["long"])
                future_to_index[future] = idx

        # Process completed tasks as they finish
        for future in as_completed(future_to_index):
            idx = future_to_index[future]

            try:
                # Store result in correct position
                results[idx] = future.result()

            except Exception:
                # In case of unexpected failure, keep as None
                results[idx] = None

    return results


# =========================
# EXECUTION
# =========================
start_time = time.time()

# Fill missing soil organic carbon values
df["org_C"] = fill_orgC_column(df)

end_time = time.time()

print(f"Completed in {(end_time - start_time)/60:.2f} minutes")


# =========================
# SAVE OUTPUT DATASET
# =========================
df.to_csv(r"C:\Python\master_dataset_filled.csv", index=False)